# BTIS3043 Artificial Intelligence — Final Assessment (2026B)
**Core task:** Query all three eBook catalogues (Dataset A, B, C) using both predicate-only and fuzzy-enhanced reasoning, for two fixed scenarios, and compare the outcomes.

This notebook is the reproducible implementation backing the technical report. It imports the reusable engine modules from `src/`:
- `src/data_loader.py` — loads the three datasets
- `src/predicate_engine.py` — crisp/Boolean predicate querying (dataset-aware field search)
- `src/fuzzy_engine.py` — fuzzy membership functions + weighted aggregation (evidence-aware)


## 1. Dataset and Knowledge Representation

This intelligent eBook search system uses three datasets provided for the BTIS3043 Final Assessment. Each dataset represents a different type of eBook collection and contains different attributes. Therefore, the datasets are processed separately instead of being merged into a common structure.

### Dataset A — Existing eBook Collection

Dataset A represents the DCS department's current eBook holdings. It contains a small number of records and provides basic information such as title, copyright year and unit net price. Since there are no detailed discipline or category fields, predicate searching for this dataset mainly depends on the **Title** field.

### Dataset B — Academic eBook Catalogue

Dataset B is a larger academic eBook catalogue containing potential reference materials. Besides the title, it provides four levels of discipline information. These discipline fields allow the system to identify relevant books even when the scenario keyword does not appear directly in the title.

Dataset B also provides copyright and eBook format information. However, it does not provide a suitable comparable price field. Therefore, affordability is not included in the fuzzy evaluation for Dataset B.

### Dataset C — eBook Acquisition Catalogue

Dataset C represents potential eBook acquisitions and includes more detailed information such as title, category, discipline, copyright year, eBook format and several licensing-price arrangements.

The **Title**, **Category** and **Discipline** fields are used for predicate searching. This allows the system to identify both directly related books and books that are relevant through their subject classification.

For affordability evaluation, the **Single user / 1-Year** licence is selected as the representative price field because it provides a consistent licensing arrangement that can be compared across records.

### Difference Between the Datasets

The datasets differ in size, structure and available evidence. Dataset A is small and has limited subject information, while Dataset B and Dataset C provide richer discipline-related attributes. These differences may affect the number of predicate matches and the usefulness of fuzzy reasoning.

The system therefore uses dataset-specific searchable fields and only applies fuzzy factors when the required evidence is available.

In [19]:
import pandas as pd

from src.data_loader import load_datasets
from src.predicate_engine import (
    predicate_query,
    classify_relevance
)
from src.fuzzy_engine import (
    build_affordability_thresholds,
    evaluate_record
)

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

# Load the three datasets
df_a, df_b, df_c = load_datasets("data")

DFS = {
    "A": df_a,
    "B": df_b,
    "C": df_c
}

print("Dataset sizes")
print("-" * 50)
print(f"Dataset A — Existing Collection:   {df_a.shape}")
print(f"Dataset B — Academic Catalogue:    {df_b.shape}")
print(f"Dataset C — Acquisition Catalogue: {df_c.shape}")

Dataset sizes
--------------------------------------------------
Dataset A — Existing Collection:   (9, 11)
Dataset B — Academic Catalogue:    (1743, 13)
Dataset C — Acquisition Catalogue: (807, 30)


## 2. Affordability Configuration

Affordability is one of the fuzzy factors used to evaluate the suitability of the returned eBooks. However, price information is not available in the same form across all three datasets.

Dataset A provides a **Unit Net Price**, while Dataset C provides several licensing-price options. For Dataset C, the **Single user / 1-Year** licence is selected as the representative price field because it provides a consistent basis for comparing acquisition cost across records.

Dataset B does not contain a suitable comparable price field. Therefore, affordability is treated as unavailable for Dataset B instead of assigning an artificial score.

The affordability membership boundaries are derived separately from the price distribution of each applicable dataset. The **25th percentile** is used as the lower-price threshold, while the **90th percentile** is used as the higher-price threshold.

A price at or below the lower threshold receives a high affordability membership value. As the price increases towards the higher threshold, the affordability membership gradually decreases. This allows affordability to be represented as a gradual preference rather than a strict Boolean condition.

Using dataset-specific thresholds is more appropriate than applying one fixed price range because Dataset A and Dataset C contain different pricing structures and values.

The selected percentile boundaries are heuristic design choices for this prototype rather than universal affordability standards. They allow affordability to be evaluated relative to the price distribution of each catalogue.

In [20]:
# Build dataset-specific affordability thresholds
affordability_thresholds = build_affordability_thresholds(
    df_a,
    df_c
)

print("Affordability thresholds")
print("-" * 50)

for dataset, config in affordability_thresholds.items():
    print(
        f"Dataset {dataset}: "
        f"Price field = '{config['field']}', "
        f"Low threshold = {config['low']:.2f}, "
        f"High threshold = {config['high']:.2f}"
    )

Affordability thresholds
--------------------------------------------------
Dataset A: Price field = 'Unit Net Price', Low threshold = 212.86, High threshold = 638.65
Dataset C: Price field = 'Single user / 1-Year', Low threshold = 86.69, High threshold = 196.44


## 3. Predicate Query Design

The predicate query component is used as the first filtering stage of the intelligent eBook search system. It applies crisp Boolean conditions to identify whether a record satisfies the defined scenario requirements.

A record either satisfies the predicate condition or does not satisfy it. Therefore, this stage does not assign gradual suitability scores. Its purpose is to reduce the search space and return records that are relevant enough to be evaluated further by the fuzzy reasoning component.

Because the three datasets contain different attributes, the searchable fields are configured separately for each dataset.

### Dataset-Specific Search Fields

- **Dataset A:** Title
- **Dataset B:** Title and Discipline (Level 1–4)
- **Dataset C:** Title, Category and Discipline

Dataset A mainly depends on title matching because it does not provide detailed subject classification fields. Dataset B and Dataset C contain richer discipline-related information, allowing relevant records to be identified through either title keywords or subject classification.

The system uses case-insensitive keyword matching. Multiple keywords can be used together, and a record is accepted when at least one relevant keyword is found in the configured searchable fields.

The predicate engine also records the keyword and field responsible for each match. This provides a simple explanation of why a record satisfied the predicate query.

Special handling is included for keywords that may cause ambiguous matches. For example, the general term **security** can appear in unrelated topics such as food security. Therefore, when the keyword `security` is used, the record must also contain computing-related context such as computer, network, information, software, cyber, digital or system terminology.

This reduces false-positive results while still allowing valid security-related titles such as **Computer Security**, **Network Security** and **Information Security** to be retrieved.

The keyword-matching method also supports terms containing special symbols such as `C++`. Instead of relying only on standard word-boundary matching, the predicate engine uses a safer regular-expression boundary so that programming keywords containing symbols can still be identified correctly.

The Scenario 1 vocabulary also includes `C Programming` to represent C-related programming support without using the overly broad single-character keyword `C`. Using only `C` could produce inaccurate matches because the letter may appear in unrelated text.

For Scenario 2, the system uses more specific terms such as `Computer Forensics` rather than the broader term `Forensics`. This helps reduce unrelated matches while still identifying records that are clearly connected to cybersecurity and secure computing.

Overall, the predicate design combines dataset-specific fields, scenario-specific keywords and contextual restrictions to improve the precision and explainability of the initial search stage.

In [21]:
# ---------------------------------------------------------
# Scenario 1:
# Artificial Intelligence, Programming and
# Mathematical Foundations
# ---------------------------------------------------------

SCENARIO1_DIRECT = [
    "Artificial Intelligence",
    "AI",
    "Intelligent Systems",
    "Machine Learning",
    "Computer Vision",
    "Robotics",
    "Expert Systems",
    "Knowledge Representation"
]

SCENARIO1_PROGRAMMING = [
    "Python",
    "Java",
    "C Programming",
    "C++",
    "Programming",
    "Algorithms",
    "Data Structures"
]

SCENARIO1_MATHEMATICS = [
    "Statistics",
    "Probability",
    "Linear Algebra",
    "Discrete Mathematics",
    "Calculus",
    "Optimization",
    "Decision Analysis"
]

SCENARIO1_KEYWORDS = (
    SCENARIO1_DIRECT
    + SCENARIO1_PROGRAMMING
    + SCENARIO1_MATHEMATICS
)

SCENARIO1_SUPPORT = {
    "Programming Support": SCENARIO1_PROGRAMMING,
    "Mathematical Support": SCENARIO1_MATHEMATICS
}


# ---------------------------------------------------------
# Scenario 2:
# Cybersecurity and Secure Computing
# ---------------------------------------------------------

SCENARIO2_DIRECT = [
    "Cybersecurity",
    "Cyber Security",
    "Computer Security",
    "Network Security",
    "Information Security",
    "Security",
    "Cryptography",
    "Cryptographic",
    "Privacy",
    "Digital Forensics",
    "Computer Forensics",
    "Information Assurance",
    "Secure Systems",
    "Secure Computing"
]

SCENARIO2_KEYWORDS = SCENARIO2_DIRECT

SCENARIO2_SUPPORT = {}


# ---------------------------------------------------------
# Display configuration
# ---------------------------------------------------------

print("Scenario 1")
print("-" * 50)
print(f"Direct AI keywords: {len(SCENARIO1_DIRECT)}")
print(f"Programming-support keywords: {len(SCENARIO1_PROGRAMMING)}")
print(f"Mathematical-support keywords: {len(SCENARIO1_MATHEMATICS)}")
print(f"Total predicate keywords: {len(SCENARIO1_KEYWORDS)}")

print("\nScenario 2")
print("-" * 50)
print(f"Security-related keywords: {len(SCENARIO2_KEYWORDS)}")

Scenario 1
--------------------------------------------------
Direct AI keywords: 8
Programming-support keywords: 7
Mathematical-support keywords: 7
Total predicate keywords: 22

Scenario 2
--------------------------------------------------
Security-related keywords: 14


## 4. Fuzzy Reasoning Design

After the predicate query identifies records that satisfy the basic scenario conditions, fuzzy reasoning is used to evaluate how suitable each returned record is.

Unlike predicate reasoning, fuzzy reasoning does not make only a yes-or-no decision. Instead, each relevant characteristic is represented using a membership value between **0 and 1**. A higher value indicates that the record satisfies that preference more strongly.

This system evaluates four fuzzy aspects:

1. **Topic Relevance**
2. **Publication Recency**
3. **eBook Format Suitability**
4. **Affordability**

The available fuzzy evidence differs between the three datasets. Therefore, a fuzzy factor is only used when the required attribute is available. Missing evidence is not automatically assigned a neutral score. Instead, the remaining available weights are re-normalised before calculating the final fuzzy suitability score.

### 4.1 Topic Relevance

Topic relevance represents how strongly a record supports the scenario.

For Scenario 1, records can be classified as directly related to Artificial Intelligence, programming support, mathematical support or another justified relationship.

The following membership values are used:

| Relationship | Membership Value |
|---|---:|
| Directly Related | 1.00 |
| Programming Support | 0.70 |
| Mathematical Support | 0.70 |
| Other Justified Match | 0.45 |

A directly related record receives the highest relevance membership because it most closely satisfies the main scenario requirement. Supporting programming and mathematical references receive lower but still meaningful membership values.

### 4.2 Publication Recency

Publication recency is treated as a gradual preference instead of a strict publication-year condition.

Very recent books receive a higher membership value, while older books receive progressively lower values.

The recency membership is interpreted as follows:

- Books published within the most recent two years receive a membership value of **1.00**.
- Books around three to five years old gradually decrease towards **0.60**.
- Books around six to ten years old gradually decrease from **0.60 to 0.30**.
- Books more than ten years old receive a lower membership value of **0.20**.

This design allows older but highly relevant books to remain in the result instead of being completely rejected.

### 4.3 eBook Format Suitability

Where eBook format information is available, the system evaluates whether the format is suitable for electronic reference use.

Common digital formats such as **EPUB, PDF, HTML and online eBook formats** receive high membership values. Other available formats whose suitability is less clearly identified receive a moderate membership value.

Dataset A does not provide a configured eBook format field, so format suitability is not evaluated for Dataset A. Dataset B and Dataset C provide eBook format information and therefore include this factor.

### 4.4 Affordability

Affordability is evaluated only for datasets containing comparable price information.

Dataset A uses **Unit Net Price**, while Dataset C uses the **Single user / 1-Year** licensing arrangement. Dataset B does not contain a suitable comparable price field, so affordability is not applied to Dataset B.

Prices at or below the lower affordability threshold receive a membership value close to **1.00**. As the price approaches the upper threshold, affordability gradually decreases. Very expensive records receive a lower membership value.

### 4.5 Fuzzy Aggregation

The default fuzzy weights are:

| Fuzzy Factor | Weight |
|---|---:|
| Topic Relevance | 0.45 |
| Publication Recency | 0.25 |
| eBook Format Suitability | 0.15 |
| Affordability | 0.15 |

Topic relevance receives the highest weight because matching the academic requirement is the most important consideration. Recency receives the second-highest weight, while format suitability and affordability are treated as additional preferences.

If one or more fuzzy factors are unavailable for a dataset, the unavailable factor is excluded and the remaining weights are re-normalised so that the total weight remains equal to 1.

The final fuzzy suitability score is calculated using a weighted average of the available fuzzy membership values. Records are then ranked from the highest fuzzy score to the lowest fuzzy score.

This allows the system to retain the crisp filtering ability of predicate reasoning while also producing a more useful ordering based on gradual preferences.

In [22]:
from src.fuzzy_engine import (
    relevance_membership,
    recency_membership,
    format_membership,
    affordability_membership,
    DEFAULT_WEIGHTS
)

print("Fuzzy Reasoning Configuration")
print("-" * 50)

print("\nDefault fuzzy weights:")
for factor, weight in DEFAULT_WEIGHTS.items():
    print(f"{factor.capitalize():15s}: {weight:.2f}")

print("\nRelevance membership examples:")
print(f"Directly Related     : {relevance_membership('Directly Related'):.2f}")
print(f"Programming Support  : {relevance_membership('Programming Support'):.2f}")
print(f"Mathematical Support : {relevance_membership('Mathematical Support'):.2f}")
print(f"Other Justified Match: {relevance_membership('Other Justified Match'):.2f}")

print("\nRecency membership examples:")
for year in [2026, 2024, 2022, 2018, 2010]:
    print(
        f"Year {year}: "
        f"{recency_membership(year):.2f}"
    )

print("\nFormat suitability examples:")
for ebook_format in ["EPUB", "PDF", "HTML", "Digital", "Other"]:
    print(
        f"{ebook_format:10s}: "
        f"{format_membership(ebook_format):.2f}"
    )

Fuzzy Reasoning Configuration
--------------------------------------------------

Default fuzzy weights:
Relevance      : 0.45
Recency        : 0.25
Format         : 0.15
Affordability  : 0.15

Relevance membership examples:
Directly Related     : 1.00
Programming Support  : 0.70
Mathematical Support : 0.70
Other Justified Match: 0.45

Recency membership examples:
Year 2026: 1.00
Year 2024: 1.00
Year 2022: 0.73
Year 2018: 0.42
Year 2010: 0.20

Format suitability examples:
EPUB      : 1.00
PDF       : 1.00
HTML      : 1.00
Digital   : 0.70
Other     : 0.50


## 5. Fixed Scenario 1 — Artificial Intelligence, Programming and Mathematical Foundations

The first fixed scenario evaluates current and potential eBook references that may support the teaching of Artificial Intelligence and intelligent systems.

The scenario considers three types of academic relationships:

1. **Directly AI-related references**
2. **Programming support**
3. **Mathematical support**

Direct AI-related keywords include Artificial Intelligence, intelligent systems, machine learning, computer vision, robotics, expert systems and knowledge representation.

Programming support includes areas such as Python, Java, C programming, C++, programming, algorithms and data structures. Mathematical support includes statistics, probability, linear algebra, discrete mathematics, calculus, optimization and decision analysis.

The predicate stage first identifies records containing at least one relevant keyword within the dataset-specific searchable fields. Each accepted record is then classified according to its relationship with the scenario.

The fuzzy stage evaluates the predicate results using topic relevance, publication recency, eBook format suitability and affordability where the required evidence is available.

For each dataset, the predicate-only results are compared with the fuzzy-enhanced ranking. Up to five suitable records are displayed, following the assessment requirement.

In [23]:
# ---------------------------------------------------------
# Helper functions used for both scenarios
# ---------------------------------------------------------

YEAR_FIELDS = {
    "A": "Copyright Year",
    "B": "Copyright",
    "C": "Copyright Year"
}


def classify_record_relationship(
    row,
    direct_keywords,
    support_map
):
    """
    Classify the relationship using the actual predicate terms
    that caused the record to match.

    This allows discipline/category matches to be classified
    correctly even when the keyword is not present in the title.
    """

    matched_text = str(
        row.get("_matched_terms", "")
    ).lower()

    # Direct relationship
    for keyword in direct_keywords:
        if str(keyword).lower() in matched_text:
            return "Directly Related"

    # Supporting relationship
    for label, keywords in support_map.items():
        for keyword in keywords:
            if str(keyword).lower() in matched_text:
                return label

    return "Other Justified Match"


def run_query(
    dataset_key,
    keywords,
    direct_keywords,
    support_map
):
    """
    Run predicate filtering followed by fuzzy evaluation
    for one dataset.
    """

    df = DFS[dataset_key]

    # Predicate-only stage
    predicate_result = predicate_query(
        df,
        dataset_key,
        keywords
    )

    if predicate_result.empty:
        return predicate_result, predicate_result

    predicate_result = predicate_result.copy()

    # Relationship classification
    predicate_result["Relevance_Label"] = (
        predicate_result.apply(
            lambda row: classify_record_relationship(
                row,
                direct_keywords,
                support_map
            ),
            axis=1
        )
    )

    # Fuzzy evaluation
    fuzzy_rows = []

    for index, row in predicate_result.iterrows():

        scores = evaluate_record(
            row=row,
            dataset_key=dataset_key,
            relationship_label=row["Relevance_Label"],
            year_field=YEAR_FIELDS[dataset_key],
            affordability_thresholds=affordability_thresholds
        )

        fuzzy_rows.append(scores)

    fuzzy_scores_df = pd.DataFrame(
        fuzzy_rows,
        index=predicate_result.index
    )

    fuzzy_result = pd.concat(
        [
            predicate_result,
            fuzzy_scores_df
        ],
        axis=1
    )

    fuzzy_result = fuzzy_result.sort_values(
        by="Fuzzy_Score",
        ascending=False
    )

    return predicate_result, fuzzy_result


# ---------------------------------------------------------
# Display columns
# ---------------------------------------------------------

DISPLAY_COLS = [
    "Title",
    "Relevance_Label",
    "_matched_terms",
    "_matched_field",
    "Relevance_Score",
    "Recency_Score",
    "Format_Score",
    "Affordability_Score",
    "Fuzzy_Score"
]


# ---------------------------------------------------------
# Run Scenario 1
# ---------------------------------------------------------

scenario1_results = {}

for dataset_key in ["A", "B", "C"]:

    predicate_result, fuzzy_result = run_query(
        dataset_key,
        SCENARIO1_KEYWORDS,
        SCENARIO1_DIRECT,
        SCENARIO1_SUPPORT
    )

    scenario1_results[dataset_key] = {
        "predicate": predicate_result,
        "fuzzy": fuzzy_result
    }

    print(
        f"Dataset {dataset_key}: "
        f"{len(predicate_result)} predicate match(es)"
    )

Dataset A: 0 predicate match(es)
Dataset B: 229 predicate match(es)
Dataset C: 108 predicate match(es)


### 5.1 Dataset A — Scenario 1

Dataset A represents the DCS department's existing eBook collection. Predicate searching is limited mainly to the **Title** field because Dataset A does not provide detailed subject or discipline information.

For Scenario 1, Dataset A returns no predicate matches. This means that none of the current titles satisfy the defined Artificial Intelligence, programming or mathematical-foundation keyword conditions.

The absence of matches is consistent with the small size and limited searchable metadata of the current collection. However, keyword-based searching may also miss relevant records that use terminology outside the defined scenario vocabulary.

Since no records satisfy the predicate stage, fuzzy evaluation cannot provide an additional ranking for Dataset A in this scenario.

In [24]:
pred_a_s1 = scenario1_results["A"]["predicate"]
fuzzy_a_s1 = scenario1_results["A"]["fuzzy"]

print("Dataset A — Scenario 1")
print("-" * 60)

if pred_a_s1.empty:

    print("No records satisfy the Scenario 1 predicate.")
    print(
        "Dataset A contains only a small existing collection "
        "and has no detailed discipline field."
    )

else:

    print("Predicate-only results:")
    display(
        pred_a_s1[
            [
                "Title",
                "_matched_terms",
                "_matched_field",
                "Relevance_Label"
            ]
        ].head(5)
    )

    print("\nFuzzy-enhanced results:")
    display(
        fuzzy_a_s1[
            [
                col
                for col in DISPLAY_COLS
                if col in fuzzy_a_s1.columns
            ]
        ].head(5)
    )

Dataset A — Scenario 1
------------------------------------------------------------
No records satisfy the Scenario 1 predicate.
Dataset A contains only a small existing collection and has no detailed discipline field.


### 5.2 Dataset B — Scenario 1

Dataset B is the largest catalogue and provides both title information and a four-level discipline hierarchy. Therefore, the predicate query can identify relevant records through either the title or discipline classification.

The predicate-only result represents all records that satisfy at least one Scenario 1 keyword condition. At this stage, every returned record is treated as equally acceptable and no suitability ranking is applied.

The fuzzy-enhanced stage evaluates each predicate result according to topic relevance, publication recency and eBook format suitability. Affordability is not evaluated because Dataset B does not provide a suitable comparable price field.

The fuzzy ranking is expected to prioritise records that are directly related to Artificial Intelligence and are relatively recent. Programming and mathematical-support references remain useful but generally receive a lower relevance membership than directly AI-related records.

The first five predicate matches are displayed in their original catalogue order and compared with the five highest-ranked fuzzy results.

In [25]:
pred_b_s1 = scenario1_results["B"]["predicate"]
fuzzy_b_s1 = scenario1_results["B"]["fuzzy"]

print("Dataset B — Scenario 1")
print("=" * 70)

print(
    f"Total predicate matches: "
    f"{len(pred_b_s1)}"
)

print("\nPredicate-only results — first 5 in catalogue order:")
display(
    pred_b_s1[
        [
            "Title",
            "_matched_terms",
            "_matched_field",
            "Relevance_Label"
        ]
    ].head(5)
)

print("\nFuzzy-enhanced results — top 5:")
display(
    fuzzy_b_s1[
        [
            col
            for col in DISPLAY_COLS
            if col in fuzzy_b_s1.columns
        ]
    ].head(5)
)

Dataset B — Scenario 1
Total predicate matches: 229

Predicate-only results — first 5 in catalogue order:


,Title,_matched_terms,_matched_field,Relevance_Label
3,"Foundations of Decision Analysis, Global Edition",Decision Analysis,Title,Mathematical Support
11,"Statistical Methods for the Social Sciences, Global Edition",Statistics,"Discipline (Level 2), Discipline (Level 3)",Mathematical Support
12,"Statistics: The Art and Science of Learning from Data, Global Edition",Statistics,"Title, Discipline (Level 2), Discipline (Level 3), Discipline (Level 4)",Mathematical Support
13,"Statistical Methods for the Social Sciences, Global Edition",Statistics,"Discipline (Level 2), Discipline (Level 3)",Mathematical Support
15,"Absolute C++, Global Edition","C++, Programming","Title, Discipline (Level 3), Discipline (Level 4)",Programming Support



Fuzzy-enhanced results — top 5:


,Title,Relevance_Label,_matched_terms,_matched_field,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score
117,Artificial Intelligence: A Guide to Intelligent Systems,Directly Related,"Artificial Intelligence, Intelligent Systems","Title, Discipline (Level 3), Discipline (Level 4)",1.0,1.0000,1.0,None,1.0000
1410,"Business Intelligence, Analytics, Data Science, and AI, Global Edition",Directly Related,AI,Title,1.0,1.0000,1.0,None,1.0000
134,"Artificial Intelligence: A Modern Approach, Global Edition\n",Directly Related,Artificial Intelligence,"Title, Discipline (Level 3), Discipline (Level 4)",1.0,0.7333,1.0,None,0.9216
92,"Statistics for Economics, Accounting and Business Studies",Mathematical Support,Statistics,Title,0.7,1.0000,1.0,None,0.8412
43,"Statistics for Psychology, Global Edition",Mathematical Support,Statistics,"Title, Discipline (Level 4)",0.7,1.0000,1.0,None,0.8412


### 5.3 Dataset C — Scenario 1

Dataset C contains title, category and discipline information together with publication year, eBook format and multiple licensing-price options. Therefore, it provides a wider range of evidence for both predicate and fuzzy reasoning.

The predicate query searches the **Title**, **Category** and **Discipline** fields. This allows records to be retrieved even when an AI, programming or mathematical keyword appears only in the subject classification rather than directly in the title.

For fuzzy evaluation, Dataset C uses all four available fuzzy factors: topic relevance, recency, format suitability and affordability.

Affordability is calculated using the **Single user / 1-Year** licence as the representative comparable price field. Therefore, two similarly relevant and recent records may receive different final rankings when one provides a more affordable licensing option.

The first five predicate-only records are compared with the five highest-ranked fuzzy-enhanced records.

In [26]:
pred_c_s1 = scenario1_results["C"]["predicate"]
fuzzy_c_s1 = scenario1_results["C"]["fuzzy"]

print("Dataset C — Scenario 1")
print("=" * 70)

print(
    f"Total predicate matches: "
    f"{len(pred_c_s1)}"
)

print("\nPredicate-only results — first 5 in catalogue order:")
display(
    pred_c_s1[
        [
            "Title",
            "_matched_terms",
            "_matched_field",
            "Relevance_Label"
        ]
    ].head(5)
)

print("\nFuzzy-enhanced results — top 5:")
display(
    fuzzy_c_s1[
        [
            col
            for col in DISPLAY_COLS
            if col in fuzzy_c_s1.columns
        ]
    ].head(5)
)

Dataset C — Scenario 1
Total predicate matches: 108

Predicate-only results — first 5 in catalogue order:


,Title,_matched_terms,_matched_field,Relevance_Label
10,Introduction Techinical Mathematics,Statistics,Category,Mathematical Support
28,Calculus,"Calculus, Statistics","Title, Category",Mathematical Support
29,"Calculus, Metric Edition","Calculus, Statistics","Title, Category",Mathematical Support
30,Calculus: Early Transcendentals,"Calculus, Statistics","Title, Category",Mathematical Support
31,"Calculus: Early Transcendentals, Metric Edition","Calculus, Statistics","Title, Category",Mathematical Support



Fuzzy-enhanced results — top 5:


,Title,Relevance_Label,_matched_terms,_matched_field,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score
163,"Artificial Intelligence, 2e",Directly Related,Artificial Intelligence,Title,1.0,0.7333,1.0,1.0000,0.9333
421,Introduction to Artificial Intelligence: A Business Perspective,Directly Related,Artificial Intelligence,Title,1.0,1.0000,1.0,0.5475,0.9321
777,"Artificial Intelligence, Analytics and Data Science (Vol. 1)",Directly Related,"Artificial Intelligence, Programming","Title, Discipline",1.0,0.6000,1.0,1.0000,0.9000
281,Discovering Mathematics: A Quantitative Reasoning Approach,Mathematical Support,Statistics,Category,0.7,1.0000,1.0,1.0000,0.8650
443,MATLAB Programming for Engineers,Programming Support,Programming,Title,0.7,1.0000,1.0,1.0000,0.8650


### 5.4 Scenario 1 Comparison

The results demonstrate a clear difference between predicate-only and fuzzy-enhanced searching.

Predicate reasoning is effective for identifying records that satisfy the defined Artificial Intelligence, programming or mathematical-support conditions. However, predicate-only results are not ranked according to overall suitability. Records remain in their original catalogue order after satisfying the Boolean conditions.

A clear ranking change can be observed in Dataset B. In the predicate-only output, *Foundations of Decision Analysis, Global Edition* appears first because it satisfies the mathematical-support predicate and occurs early in the catalogue. However, after fuzzy evaluation, *Artificial Intelligence: A Guide to Intelligent Systems* moves to the top of the results because it is directly related to the main scenario and receives strong relevance, recency and format membership values.

This demonstrates that fuzzy reasoning changes the result from a collection of acceptable records into a prioritised list based on gradual preferences.

Dataset A produces no Scenario 1 matches because its existing collection is small and contains limited searchable subject evidence. The result also highlights a limitation of keyword-based searching because relevant books may be missed if different terminology is used.

Dataset B produces substantially more matches because it is a much larger catalogue and provides four levels of discipline information. Its fuzzy ranking is based mainly on topic relevance, recency and format suitability because comparable price evidence is unavailable.

Dataset C also returns many relevant records and contains richer evidence for fuzzy evaluation. In addition to relevance, recency and format, affordability can influence the ranking because licensing-price information is available.

Therefore, fuzzy reasoning provides the greatest practical benefit when the predicate stage returns a large number of candidate records. Instead of requiring the department to review every matching record manually, the fuzzy stage produces a more useful prioritised shortlist.

In [27]:
scenario1_summary = []

for dataset_key in ["A", "B", "C"]:

    pred = scenario1_results[dataset_key]["predicate"]
    fuzzy = scenario1_results[dataset_key]["fuzzy"]

    scenario1_summary.append(
        {
            "Dataset": dataset_key,
            "Predicate Matches": len(pred),
            "Top Fuzzy Score": (
                fuzzy["Fuzzy_Score"].max()
                if not fuzzy.empty
                else None
            ),
            "Format Evidence": (
                "Yes"
                if dataset_key in ["B", "C"]
                else "No"
            ),
            "Affordability Evidence": (
                "Yes"
                if dataset_key in ["A", "C"]
                else "No"
            )
        }
    )

scenario1_summary_df = pd.DataFrame(
    scenario1_summary
)

display(scenario1_summary_df)

,Dataset,Predicate Matches,Top Fuzzy Score,Format Evidence,Affordability Evidence
0,A,0,NaN,No,Yes
1,B,229,1.0000,Yes,No
2,C,108,0.9333,Yes,Yes


## 6. Fixed Scenario 2 — Cybersecurity and Secure Computing

The second fixed scenario evaluates existing and potential reference eBooks related to cybersecurity and secure computing.

Relevant subject areas include cybersecurity, computer security, network security, information security, cryptography, privacy, digital forensics, information assurance, secure systems and related secure-computing topics.

The predicate stage identifies records containing the selected security-related terms within each dataset's configured searchable fields.

A special contextual rule is applied to the general keyword **security**. The term `security` alone is too broad because it may occur in unrelated subjects such as food security or energy security. Therefore, a general security match is accepted only when computing-related context such as computer, cyber, network, information, software, system or digital terminology is also present.

This improves predicate precision and prevents unrelated records from entering the fuzzy-ranking stage.

For fuzzy evaluation, records are ranked according to topic relevance, publication recency, eBook format suitability and affordability where comparable price information exists.

All relevant Dataset A records are displayed because Dataset A represents the current collection. For the larger candidate catalogues, up to ten relevant records are displayed.

In [28]:
# Run Scenario 2 for all datasets

scenario2_results = {}

for dataset_key in ["A", "B", "C"]:

    predicate_result, fuzzy_result = run_query(
        dataset_key,
        SCENARIO2_KEYWORDS,
        SCENARIO2_DIRECT,
        SCENARIO2_SUPPORT
    )

    scenario2_results[dataset_key] = {
        "predicate": predicate_result,
        "fuzzy": fuzzy_result
    }

    print(
        f"Dataset {dataset_key}: "
        f"{len(predicate_result)} predicate match(es)"
    )

Dataset A: 1 predicate match(es)
Dataset B: 9 predicate match(es)
Dataset C: 6 predicate match(es)


### 6.1 Dataset A — Scenario 2

Dataset A represents the existing DCS eBook collection. Because the assessment requires all relevant current-subscription records to be presented, every Dataset A record satisfying the cybersecurity predicate is shown.

The predicate query identifies security-related titles from the current collection. Since Dataset A contains price information but no configured eBook format field, its fuzzy evaluation uses topic relevance, publication recency and affordability.

The result provides an indication of how much cybersecurity coverage already exists within the current collection before additional acquisitions are considered.

In [29]:
pred_a_s2 = scenario2_results["A"]["predicate"]
fuzzy_a_s2 = scenario2_results["A"]["fuzzy"]

print("Dataset A — Scenario 2")
print("=" * 70)

print(
    f"Total relevant current records: "
    f"{len(pred_a_s2)}"
)

if pred_a_s2.empty:

    print("No cybersecurity-related current records were found.")

else:

    print("\nPredicate-only results:")
    display(
        pred_a_s2[
            [
                "Title",
                "_matched_terms",
                "_matched_field",
                "Relevance_Label"
            ]
        ]
    )

    print("\nFuzzy-enhanced results:")
    display(
        fuzzy_a_s2[
            [
                col
                for col in DISPLAY_COLS
                if col in fuzzy_a_s2.columns
            ]
        ]
    )

Dataset A — Scenario 2
Total relevant current records: 1

Predicate-only results:


,Title,_matched_terms,_matched_field,Relevance_Label
5,Security in Computing,Security,Title,Directly Related



Fuzzy-enhanced results:


,Title,Relevance_Label,_matched_terms,_matched_field,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score
5,Security in Computing,Directly Related,Security,Title,1.0,1.0,None,0.1,0.8412


### 6.2 Dataset B — Scenario 2

Dataset B searches cybersecurity-related terminology across both the title and four-level discipline hierarchy.

Because Dataset B contains no suitable comparable price field, affordability is excluded from its fuzzy evaluation. The remaining available factors are automatically re-weighted so that topic relevance, publication recency and eBook format suitability determine the final fuzzy score.

The predicate-only stage identifies all records satisfying the security conditions but does not distinguish between stronger and weaker candidates. The fuzzy-enhanced ranking provides a more useful ordering by prioritising suitable and recent cybersecurity references.

Up to ten records are displayed according to the assessment requirement.

In [30]:
pred_b_s2 = scenario2_results["B"]["predicate"]
fuzzy_b_s2 = scenario2_results["B"]["fuzzy"]

print("Dataset B — Scenario 2")
print("=" * 70)

print(
    f"Total predicate matches: "
    f"{len(pred_b_s2)}"
)

print("\nPredicate-only results:")
display(
    pred_b_s2[
        [
            "Title",
            "_matched_terms",
            "_matched_field",
            "Relevance_Label"
        ]
    ].head(10)
)

print("\nFuzzy-enhanced results — ranked:")
display(
    fuzzy_b_s2[
        [
            col
            for col in DISPLAY_COLS
            if col in fuzzy_b_s2.columns
        ]
    ].head(10)
)

Dataset B — Scenario 2
Total predicate matches: 9

Predicate-only results:


,Title,_matched_terms,_matched_field,Relevance_Label
200,"Boyle: Corporate Computer Security, Global Edition","Computer Security, Security, Network Security","Title, Discipline (Level 4)",Directly Related
421,"Computer Security: Principles and Practice, Global Edition","Computer Security, Security",Title,Directly Related
422,"Computer Security: Principles and Practice, Global Edition","Computer Security, Security","Title, Discipline (Level 3), Discipline (Level 4)",Directly Related
430,"Cryptography and Network Security: Principles and Practice, Global Edition","Network Security, Security, Cryptography, Computer Security","Title, Discipline (Level 3), Discipline (Level 4)",Directly Related
848,Introduction to Computer Security,"Computer Security, Security",Title,Directly Related
1228,"Business Data Networks and Security, Global Edition",Security,Title,Directly Related
1308,"Network Security Essentials: Applications and Standards, Global Edition","Network Security, Security, Computer Security","Title, Discipline (Level 3), Discipline (Level 4)",Directly Related
1389,Practical Cryptology and Web Security,Security,Title,Directly Related
1741,Security in Computing,Security,Title,Directly Related



Fuzzy-enhanced results — ranked:


,Title,Relevance_Label,_matched_terms,_matched_field,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score
421,"Computer Security: Principles and Practice, Global Edition",Directly Related,"Computer Security, Security",Title,1.0,1.0000,1.0,None,1.0000
1741,Security in Computing,Directly Related,Security,Title,1.0,1.0000,1.0,None,1.0000
430,"Cryptography and Network Security: Principles and Practice, Global Edition",Directly Related,"Network Security, Security, Cryptography, Computer Security","Title, Discipline (Level 3), Discipline (Level 4)",1.0,0.7333,0.5,None,0.8333
1308,"Network Security Essentials: Applications and Standards, Global Edition",Directly Related,"Network Security, Security, Computer Security","Title, Discipline (Level 3), Discipline (Level 4)",1.0,0.4800,0.5,None,0.7588
422,"Computer Security: Principles and Practice, Global Edition",Directly Related,"Computer Security, Security","Title, Discipline (Level 3), Discipline (Level 4)",1.0,0.4200,0.5,None,0.7412
200,"Boyle: Corporate Computer Security, Global Edition",Directly Related,"Computer Security, Security, Network Security","Title, Discipline (Level 4)",1.0,0.2000,0.5,None,0.6765
848,Introduction to Computer Security,Directly Related,"Computer Security, Security",Title,1.0,0.2000,0.5,None,0.6765
1228,"Business Data Networks and Security, Global Edition",Directly Related,Security,Title,1.0,0.2000,0.5,None,0.6765
1389,Practical Cryptology and Web Security,Directly Related,Security,Title,1.0,0.2000,0.5,None,0.6765


### 6.3 Dataset C — Scenario 2

Dataset C searches cybersecurity-related terms across the **Title**, **Category** and **Discipline** fields.

The dataset provides sufficient evidence for all four fuzzy factors. Topic relevance identifies the strength of the security relationship, recency reflects publication age, format suitability evaluates the available eBook format, and affordability uses the **Single user / 1-Year** licence price.

The revised security predicate also reduces false-positive matches. In particular, a record containing the phrase **food security** is no longer accepted simply because it contains the word `security`. The general security term must occur together with relevant computing context.

This demonstrates how a more carefully designed predicate can improve the quality of records entering the fuzzy stage.

The fuzzy results also demonstrate the effect of affordability on ranking. Several Dataset C records receive the same full relevance, recency and format memberships but obtain different final fuzzy scores because their Single user / 1-Year licence prices produce different affordability memberships.

For example, *Security Awareness: Applying Practical Cybersecurity in Your World* receives a final fuzzy score of 1.00 because it receives full membership for relevance, recency, format suitability and affordability. *Management of Cybersecurity* has the same relevance, recency and format memberships, but its lower affordability membership reduces its final fuzzy score. This shows how fuzzy reasoning can distinguish between records that would otherwise appear equally suitable under predicate reasoning.

Up to ten records are displayed, although fewer may appear if the catalogue contains fewer qualifying cybersecurity titles.

In [31]:
pred_c_s2 = scenario2_results["C"]["predicate"]
fuzzy_c_s2 = scenario2_results["C"]["fuzzy"]

print("Dataset C — Scenario 2")
print("=" * 70)

print(
    f"Total predicate matches: "
    f"{len(pred_c_s2)}"
)

print(
    "Selected affordability field: "
    "'Single user / 1-Year'"
)

print("\nPredicate-only results:")
display(
    pred_c_s2[
        [
            "Title",
            "_matched_terms",
            "_matched_field",
            "Relevance_Label"
        ]
    ].head(10)
)

print("\nFuzzy-enhanced results — ranked:")
display(
    fuzzy_c_s2[
        [
            col
            for col in DISPLAY_COLS
            if col in fuzzy_c_s2.columns
        ]
    ].head(10)
)

Dataset C — Scenario 2
Total predicate matches: 6
Selected affordability field: 'Single user / 1-Year'

Predicate-only results:


,Title,_matched_terms,_matched_field,Relevance_Label
228,CompTIA CySA+ Guide to Cybersecurity Analyst (CS0-003),"Cybersecurity, Security",Title,Directly Related
231,CompTIA Security+ Guide to Network Security Fundamentals,"Network Security, Security",Title,Directly Related
330,Guide to Computer Forensics and Investigations,Computer Forensics,Title,Directly Related
441,Management of Cybersecurity,"Cybersecurity, Security",Title,Directly Related
667,Principles of Information Security,"Information Security, Security",Title,Directly Related
677,Security Awareness: Applying Practical Cybersecurity in Your World,"Cybersecurity, Security",Title,Directly Related



Fuzzy-enhanced results — ranked:


,Title,Relevance_Label,_matched_terms,_matched_field,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score
677,Security Awareness: Applying Practical Cybersecurity in Your World,Directly Related,"Cybersecurity, Security",Title,1.0,1.0000,1.0,1.0000,1.0000
441,Management of Cybersecurity,Directly Related,"Cybersecurity, Security",Title,1.0,1.0000,1.0,0.7464,0.9620
231,CompTIA Security+ Guide to Network Security Fundamentals,Directly Related,"Network Security, Security",Title,1.0,1.0000,1.0,0.4182,0.9127
228,CompTIA CySA+ Guide to Cybersecurity Analyst (CS0-003),Directly Related,"Cybersecurity, Security",Title,1.0,1.0000,1.0,0.4182,0.9127
330,Guide to Computer Forensics and Investigations,Directly Related,Computer Forensics,Title,1.0,1.0000,1.0,0.4182,0.9127
667,Principles of Information Security,Directly Related,"Information Security, Security",Title,1.0,0.7333,1.0,0.4182,0.8461


### 6.4 Predicate Precision Check

A broad keyword such as `security` may create false-positive matches if it is used without contextual restrictions.

An earlier version of the predicate query could retrieve the title **Chinese Rice Bowl: Understanding Food Security in China** because the title contains the word `security`. However, the book is unrelated to cybersecurity.

The revised predicate engine applies an additional computing-context condition to the general `security` keyword. Therefore, unrelated expressions such as food security are rejected before fuzzy evaluation.

This is important because fuzzy reasoning should rank relevant predicate results rather than attempt to correct clearly irrelevant records that should have been excluded during the predicate stage.

In [32]:
# Verify that the known food-security false positive
# is no longer returned by Scenario 2.

food_security_check = pred_c_s2[
    pred_c_s2["Title"].str.contains(
        "Food Security",
        case=False,
        na=False
    )
]

print("Scenario 2 false-positive check")
print("-" * 60)

if food_security_check.empty:

    print(
        "PASS: No unrelated 'Food Security' title "
        "was returned by the cybersecurity predicate."
    )

else:

    print(
        "WARNING: A possible unrelated security record "
        "is still present:"
    )

    display(
        food_security_check[
            [
                "Title",
                "_matched_terms",
                "_matched_field"
            ]
        ]
    )

Scenario 2 false-positive check
------------------------------------------------------------
PASS: No unrelated 'Food Security' title was returned by the cybersecurity predicate.


## 7. Predicate–Fuzzy and Cross-Dataset Comparison

The two scenarios demonstrate that predicate reasoning and fuzzy reasoning perform different but complementary functions.

Predicate reasoning provides a clear Boolean filtering mechanism. A record is either accepted because it satisfies at least one defined condition or rejected because it does not. This method is transparent and easy to explain, but it does not distinguish between different levels of suitability after records have passed the predicate filter.

Fuzzy reasoning adds a second decision layer by evaluating gradual preferences. Relevant records may differ in topic strength, publication recency, format suitability and affordability. The fuzzy score therefore provides a more useful ordering for selecting references from a large candidate set.

### Effect of Dataset Size

Dataset size strongly affects the number of records available for retrieval. Dataset A contains only **9 records**, Dataset B contains **1,743 records**, and Dataset C contains **807 records**.

In Scenario 1, Dataset A returns no predicate matches, while Dataset B and Dataset C return substantially larger candidate sets. This shows that a larger catalogue increases the opportunity to identify suitable references.

However, a larger dataset does not automatically guarantee better individual records. It mainly increases the number of alternatives available for evaluation.

### Effect of Dataset Structure

Dataset structure also affects retrieval quality.

Dataset A mainly relies on the title because it does not contain detailed discipline information. A relevant book whose title does not contain an exact scenario keyword may therefore be missed.

Dataset B provides four discipline levels, while Dataset C provides Category and Discipline information. These additional attributes allow predicate searching to identify relevant records through subject classification even when the keyword does not appear directly in the title.

Therefore, richer metadata creates additional evidence for predicate reasoning and improves the ability of the system to identify related academic materials.

### Effect of Available Fuzzy Evidence

The fuzzy design is also affected by differences in dataset attributes.

Dataset A provides recency and price evidence but no configured eBook-format field. Dataset B provides recency and eBook-format evidence but does not contain a suitable comparable price field. Dataset C provides recency, format and comparable licensing-price evidence.

When a fuzzy factor is unavailable, it is excluded and the remaining weights are re-normalised. Therefore, the fuzzy scores are mainly useful for ranking records within each dataset rather than making absolute score comparisons between different datasets.

For example, a score of 1.00 in Dataset B is calculated using a different combination of available evidence from a score of 1.00 in Dataset C because Dataset B does not include affordability.

### Predicate-Only Versus Fuzzy-Enhanced Results

The benefit of fuzzy reasoning is most apparent when the predicate query returns many candidates.

For Scenario 1 Dataset B, the predicate-only result begins with *Foundations of Decision Analysis, Global Edition*. After fuzzy evaluation, directly AI-related titles such as *Artificial Intelligence: A Guide to Intelligent Systems* move to the top because they receive stronger overall suitability memberships.

For Scenario 2 Dataset C, several records are directly security-related and have similar recency and format scores. Affordability then helps distinguish the records. *Security Awareness: Applying Practical Cybersecurity in Your World* receives a higher final score because it also has stronger affordability membership.

These examples show that fuzzy reasoning improves the practical usefulness of predicate results by creating a ranked shortlist rather than simply returning an unranked set of Boolean matches.

However, fuzzy reasoning is not a replacement for good predicate design. If an irrelevant record is allowed through an overly broad predicate, fuzzy scoring may still assign it a suitability score. The improvement made to the Scenario 2 `security` predicate demonstrates why the quality of the initial filtering stage remains important.

In [33]:
comparison_rows = []

for scenario_name, scenario_results in [
    ("Scenario 1", scenario1_results),
    ("Scenario 2", scenario2_results)
]:

    for dataset_key in ["A", "B", "C"]:

        pred = scenario_results[
            dataset_key
        ]["predicate"]

        fuzzy = scenario_results[
            dataset_key
        ]["fuzzy"]

        comparison_rows.append(
            {
                "Scenario": scenario_name,
                "Dataset": dataset_key,
                "Dataset Size": len(
                    DFS[dataset_key]
                ),
                "Predicate Matches": len(pred),
                "Top Fuzzy Score": (
                    round(
                        fuzzy["Fuzzy_Score"].max(),
                        4
                    )
                    if not fuzzy.empty
                    else None
                ),
                "Format Evidence": (
                    "Yes"
                    if dataset_key in ["B", "C"]
                    else "No"
                ),
                "Affordability Evidence": (
                    "Yes"
                    if dataset_key in ["A", "C"]
                    else "No"
                )
            }
        )

comparison_df = pd.DataFrame(
    comparison_rows
)

display(comparison_df)

,Scenario,Dataset,Dataset Size,Predicate Matches,Top Fuzzy Score,Format Evidence,Affordability Evidence
0,Scenario 1,A,9,0,NaN,No,Yes
1,Scenario 1,B,1743,229,1.0000,Yes,No
2,Scenario 1,C,807,108,0.9333,Yes,Yes
3,Scenario 2,A,9,1,0.8412,No,Yes
4,Scenario 2,B,1743,9,1.0000,Yes,No
5,Scenario 2,C,807,6,1.0000,Yes,Yes


## 8. Explanation of Selected System Decisions

The system provides simple decision evidence for the selected records rather than presenting only a final score.

For each predicate result, `_matched_terms` identifies the keyword conditions that were satisfied, while `_matched_field` identifies whether the match occurred in the title, discipline or category information.

The fuzzy-enhanced output also displays the individual membership values for relevance, recency, format suitability and affordability. This makes it possible to explain why one record ranks above another.

For example, a highly ranked record may have:

- a direct relationship to the scenario;
- a recent publication year;
- a suitable eBook format; and
- a relatively affordable licence price.

Where a component is unavailable, the missing evidence is excluded rather than replaced with an artificial value. The system therefore provides a traceable connection between the original data, predicate decision, fuzzy membership values and final ranking.

In [34]:
def explain_top_records(
    fuzzy_df,
    dataset_name,
    n=3
):
    """
    Print simple explanations for the highest-ranked records.
    """

    print(
        f"\nTop decision explanations — Dataset {dataset_name}"
    )
    print("=" * 75)

    if fuzzy_df.empty:
        print("No records available.")
        return

    for rank, (_, row) in enumerate(
        fuzzy_df.head(n).iterrows(),
        start=1
    ):

        print(
            f"\nRank {rank}: {row['Title']}"
        )

        print(
            f"Matched term(s): "
            f"{row.get('_matched_terms', '')}"
        )

        print(
            f"Matched field(s): "
            f"{row.get('_matched_field', '')}"
        )

        print(
            f"Relationship: "
            f"{row.get('Relevance_Label', '')}"
        )

        print(
            f"Relevance = "
            f"{row.get('Relevance_Score')}"
        )

        print(
            f"Recency = "
            f"{row.get('Recency_Score')}"
        )

        print(
            f"Format = "
            f"{row.get('Format_Score')}"
        )

        print(
            f"Affordability = "
            f"{row.get('Affordability_Score')}"
        )

        print(
            f"Final fuzzy score = "
            f"{row.get('Fuzzy_Score')}"
        )


# Selected examples from the two scenarios
explain_top_records(
    scenario1_results["B"]["fuzzy"],
    "B — Scenario 1"
)

explain_top_records(
    scenario2_results["C"]["fuzzy"],
    "C — Scenario 2"
)


Top decision explanations — Dataset B — Scenario 1

Rank 1: Artificial Intelligence: A Guide to Intelligent Systems
Matched term(s): Artificial Intelligence, Intelligent Systems
Matched field(s): Title, Discipline (Level 3), Discipline (Level 4)
Relationship: Directly Related
Relevance = 1.0
Recency = 1.0
Format = 1.0
Affordability = None
Final fuzzy score = 1.0

Rank 2: Business Intelligence, Analytics, Data Science, and AI, Global Edition
Matched term(s): AI
Matched field(s): Title
Relationship: Directly Related
Relevance = 1.0
Recency = 1.0
Format = 1.0
Affordability = None
Final fuzzy score = 1.0

Rank 3: Artificial Intelligence: A Modern Approach, Global Edition

Matched term(s): Artificial Intelligence
Matched field(s): Title, Discipline (Level 3), Discipline (Level 4)
Relationship: Directly Related
Relevance = 1.0
Recency = 0.7333
Format = 1.0
Affordability = None
Final fuzzy score = 0.9216

Top decision explanations — Dataset C — Scenario 2

Rank 1: Security Awareness: Applyin

## 9. Strengths, Limitations and Possible Improvements

### Strengths of Predicate Reasoning

Predicate reasoning is transparent, deterministic and easy to reproduce. The system clearly defines which fields and keywords are used, making it possible to explain why a record is accepted or rejected.

It is also useful as an initial filtering stage because clearly unrelated catalogue records can be removed before more detailed fuzzy evaluation is performed.

The use of dataset-specific searchable fields improves flexibility because each catalogue can be queried according to the evidence it actually provides.

### Limitations of Predicate Reasoning

Keyword-based predicates depend heavily on the available metadata and selected vocabulary.

A relevant record may be missed if its title or discipline information uses different terminology from the defined keyword list. This problem is more significant in Dataset A because only the title provides meaningful topic evidence.

Broad keywords may also create false-positive matches. The Scenario 2 `security` example demonstrates that a word may have different meanings in different academic contexts. Additional contextual conditions are therefore necessary for ambiguous terms.

The current prototype also does not perform semantic synonym expansion. For example, a relevant concept expressed using terminology outside the selected scenario vocabulary may not be retrieved.

### Strengths of Fuzzy Reasoning

Fuzzy reasoning allows gradual preferences to be represented rather than forcing every condition into a strict Boolean rule.

This is useful for factors such as recency and affordability because there is usually no single publication year or price at which a book suddenly changes from suitable to unsuitable.

Fuzzy ranking also transforms a large collection of predicate matches into a more manageable prioritised list.

The system additionally handles missing fuzzy evidence by excluding unavailable components and re-normalising the remaining weights instead of assigning artificial neutral scores.

### Limitations of Fuzzy Reasoning

The selected membership functions and weights are based on design judgement. Different departmental priorities could lead to different weighting decisions.

In addition, not every dataset provides the same evidence. Therefore, fuzzy scores across datasets should not be interpreted as perfectly identical measurements. The fuzzy scores are mainly intended for ranking records within each dataset rather than making absolute score comparisons between Dataset A, Dataset B and Dataset C.

The current eBook format membership function is intentionally simple. Common digital formats are treated as similarly suitable, although a more detailed implementation could distinguish accessibility, offline availability, device compatibility or platform restrictions.

The affordability boundaries are also heuristic and are based on the price distributions of the available catalogues rather than a formally defined departmental budget.

Fuzzy reasoning cannot fully repair a poor predicate decision. An irrelevant record should ideally be rejected before fuzzy evaluation.

Some catalogue records may also represent duplicate or overlapping editions. Duplicate detection is not implemented in the current prototype because it is outside the minimum requirements of this assessment.

### Possible Improvements

Several improvements could be considered in future development:

1. Expand the scenario vocabulary using controlled subject terms and additional synonyms.
2. Use stronger phrase and contextual co-occurrence rules for ambiguous keywords.
3. Add semantic or discipline-match strength instead of relying mainly on keyword occurrence.
4. Allow DCS staff to adjust fuzzy weights according to teaching priorities.
5. Develop more detailed eBook format suitability criteria based on accessibility, compatibility and availability.
6. Allow affordability thresholds to reflect an actual departmental acquisition budget.
7. Evaluate additional Dataset C licensing arrangements when multiple concurrent users are required.
8. Add duplicate or related-title detection for repeated editions or overlapping catalogue records.
9. Add an explicit tie-breaking rule when multiple records receive identical fuzzy scores.

These improvements are not required for the current prototype but could increase precision, flexibility and practical usefulness.

In [35]:
print("Prototype evaluation completed.")
print("-" * 60)

print(
    "Scenario 1 total predicate matches:",
    sum(
        len(
            scenario1_results[key]["predicate"]
        )
        for key in ["A", "B", "C"]
    )
)

print(
    "Scenario 2 total predicate matches:",
    sum(
        len(
            scenario2_results[key]["predicate"]
        )
        for key in ["A", "B", "C"]
    )
)

print(
    "\nAll three datasets were queried "
    "for both required scenarios."
)

Prototype evaluation completed.
------------------------------------------------------------
Scenario 1 total predicate matches: 337
Scenario 2 total predicate matches: 16

All three datasets were queried for both required scenarios.


## 10. Conclusion

This prototype demonstrates the combined use of predicate reasoning and fuzzy reasoning for searching and evaluating academic eBooks across three structurally different catalogues.

Predicate reasoning provides the first decision stage by identifying records that satisfy the defined scenario conditions. Its main strengths are transparency, reproducibility and the ability to use dataset-specific searchable fields. However, the results also demonstrate that predicate performance depends strongly on the available metadata and the quality of the selected keyword conditions.

Fuzzy reasoning improves the predicate-only result by evaluating gradual preferences and ranking the accepted records. Topic relevance is given the highest priority, while publication recency, eBook format suitability and affordability provide additional evidence where the required attributes are available.

The two scenarios demonstrate that fuzzy reasoning is most useful when multiple predicate matches must be prioritised. In Scenario 1 Dataset B, directly AI-related records move above earlier mathematical-support predicate matches after fuzzy evaluation. In Scenario 2 Dataset C, affordability helps distinguish between records that otherwise have similar relevance, recency and format memberships.

The comparison between the datasets also shows that dataset size and metadata richness have a major effect on the outcome. Dataset A contains only nine current records and provides limited subject evidence. Dataset B provides a much larger catalogue together with detailed discipline information, while Dataset C provides discipline, format and licensing-price evidence that supports a broader fuzzy evaluation.

The implementation also demonstrates that fuzzy reasoning should operate after a meaningful predicate filter. The revised Scenario 2 security rule successfully removes the unrelated food-security record before fuzzy evaluation, showing that predicate precision and fuzzy suitability must work together.

Overall, the combined approach provides a more useful academic eBook search system than predicate-only filtering. Predicate reasoning answers **which records satisfy the required conditions**, while fuzzy reasoning evaluates **to what degree those records are suitable**. The final prototype therefore provides transparent filtering, evidence-based ranking and a simple explanation of system decisions for both required academic reference scenarios.

In [36]:
# Final reproducibility check

required_results = {
    "Scenario 1 - Dataset A": scenario1_results["A"],
    "Scenario 1 - Dataset B": scenario1_results["B"],
    "Scenario 1 - Dataset C": scenario1_results["C"],
    "Scenario 2 - Dataset A": scenario2_results["A"],
    "Scenario 2 - Dataset B": scenario2_results["B"],
    "Scenario 2 - Dataset C": scenario2_results["C"]
}

print("Final reproducibility check")
print("=" * 70)

for name, result in required_results.items():

    pred_count = len(
        result["predicate"]
    )

    fuzzy_count = len(
        result["fuzzy"]
    )

    print(
        f"{name:<28} "
        f"Predicate = {pred_count:<4} "
        f"Fuzzy evaluated = {fuzzy_count}"
    )

print("\n✓ Both fixed scenarios completed.")
print("✓ Dataset A, B and C included in both scenarios.")
print("✓ Predicate-only results generated.")
print("✓ Fuzzy-enhanced results generated.")
print("✓ Ranking and explanation evidence generated.")

Final reproducibility check
Scenario 1 - Dataset A       Predicate = 0    Fuzzy evaluated = 0
Scenario 1 - Dataset B       Predicate = 229  Fuzzy evaluated = 229
Scenario 1 - Dataset C       Predicate = 108  Fuzzy evaluated = 108
Scenario 2 - Dataset A       Predicate = 1    Fuzzy evaluated = 1
Scenario 2 - Dataset B       Predicate = 9    Fuzzy evaluated = 9
Scenario 2 - Dataset C       Predicate = 6    Fuzzy evaluated = 6

✓ Both fixed scenarios completed.
✓ Dataset A, B and C included in both scenarios.
✓ Predicate-only results generated.
✓ Fuzzy-enhanced results generated.
✓ Ranking and explanation evidence generated.
